# 03 — Notebook Proxy: Run a Chat App

The OpenShift AI notebook proxy lets you run a web app inside your notebook container and access it through a special URL — no separate deployment needed.

We will build a tiny [Chainlit](https://chainlit.io) chat app and access it at port 8080 via the proxy.

Run each cell from top to bottom by pressing **Shift + Enter**.

In [ ]:
import os
os.environ["UV_EXTRA_INDEX_URL"] = "https://pypi.org/simple"

# enable colour output for all print() calls in this notebook
!uv pip install rich -q
from rich import print, print_json

## Step 1 — Install Chainlit

In [ ]:
!uv pip install chainlit openai httpx websockets uvicorn -q

## Step 2 — Create the chat app

The `%%writefile` magic writes the cell content directly to a file.
The app reads the MaaS token from an environment variable set in Step 4, and streams the model's reply back token by token.

In [ ]:
%%writefile app.py
import os
import chainlit as cl
from openai import OpenAI

MAAS_URL = "https://maas.apps.ocp.cloud.rhai-tmm.dev/prelude-maas/qwen36-27b/v1"

@cl.on_message
async def on_message(message: cl.Message):
    client = OpenAI(base_url=MAAS_URL, api_key=os.environ.get("MAAS_TOKEN", ""))
    stream = client.chat.completions.create(
        model="qwen36-27b",
        messages=[{"role": "user", "content": message.content}],
        stream=True
    )
    reply = cl.Message(content="")
    await reply.send()
    for chunk in stream:
        if chunk.choices:
            token = chunk.choices[0].delta.content or ""
            await reply.stream_token(token)
    await reply.update()

In [ ]:
%%writefile proxy.py
"""
Reverse proxy: port 8080 (notebook proxy) → chainlit on 8081.
Prepends ROOT_PATH to every request so chainlit's routes match.
"""
import asyncio, os
import httpx, websockets
from starlette.requests import Request
from starlette.responses import Response
from starlette.websockets import WebSocket
import uvicorn

PREFIX = os.environ.get("ROOT_PATH", "")
HTTP   = "http://localhost:8081"
WS     = "ws://localhost:8081"
SKIP   = {"host", "content-length", "transfer-encoding", "content-encoding"}

async def proxy_http(request: Request) -> Response:
    path = PREFIX + (request.url.path or "/")
    url  = HTTP + path + (f"?{request.url.query}" if request.url.query else "")
    hdrs = {k: v for k, v in request.headers.items() if k.lower() not in SKIP}
    try:
        async with httpx.AsyncClient() as c:
            r = await c.request(request.method, url, headers=hdrs, content=await request.body())
        return Response(r.content, r.status_code,
                        {k: v for k, v in r.headers.items() if k.lower() not in SKIP})
    except Exception:
        return Response(b"Starting up, please retry in a moment.", 503)

async def proxy_ws(ws: WebSocket):
    await ws.accept()
    path = PREFIX + (ws.url.path or "/")
    url  = WS + path + (f"?{ws.url.query}" if ws.url.query else "")
    try:
        async with websockets.connect(url) as back:
            async def to_back():
                async for m in ws.iter_text():
                    await back.send(m)
            async def to_front():
                async for m in back:
                    await ws.send_text(m)
            done, pending = await asyncio.wait(
                [asyncio.create_task(to_back()), asyncio.create_task(to_front())],
                return_when=asyncio.FIRST_COMPLETED)
            for t in pending:
                t.cancel()
    except Exception:
        pass

class ProxyApp:
    async def __call__(self, scope, receive, send):
        try:
            if scope["type"] == "websocket":
                await proxy_ws(WebSocket(scope, receive, send))
            elif scope["type"] == "http":
                response = await proxy_http(Request(scope, receive))
                await response(scope, receive, send)
            elif scope["type"] == "lifespan":
                while True:
                    event = await receive()
                    if "startup"  in event["type"]: await send({"type": "lifespan.startup.complete"})
                    if "shutdown" in event["type"]: await send({"type": "lifespan.shutdown.complete"}); return
        except Exception:
            pass

app = ProxyApp()
if __name__ == "__main__":
    uvicorn.run("proxy:app", host="0.0.0.0", port=8080, log_level="critical")

### Why do we need a proxy?

When you open the app URL, the request travels through **three hops** before reaching Chainlit:

```
Your browser
    │
    ▼  https://host/notebook/foo/demo/proxy/8080/
OpenShift AI notebook proxy               (strips the /notebook/foo/demo/proxy/8080 prefix)
    │
    ▼  http://localhost:8080/
proxy.py                                  (adds the prefix back)
    │
    ▼  http://localhost:8081/notebook/foo/demo/proxy/8080/
Chainlit
```

**The problem** is that the OpenShift AI notebook proxy *strips* the URL prefix before forwarding the request. Chainlit needs that prefix to know where it is running, so it can tell the browser where to load its CSS and JavaScript files.

**`proxy.py` is the fix** — it sits on port 8080, receives the stripped request, adds the prefix back, and passes the full URL to Chainlit on port 8081. Without it, the browser cannot load Chainlit's page assets and you get a blank screen.

## Step 3 — Find your proxy URL

`NB_PREFIX` is set automatically by OpenShift AI and contains the path to your notebook (e.g. `/notebook/myproject/my-workbench`).
The proxy URL is that path with `/proxy/8080` appended.

In [ ]:
import os

NB_PREFIX = os.environ.get("NB_PREFIX", "")
ROOT_PATH = f"{NB_PREFIX}/proxy/8080"

print("Root path:", ROOT_PATH)
print()
print("Your proxy URL:")
print(f"  https://<your-notebook-host>{ROOT_PATH}")
print()
print("Replace <your-notebook-host> with the hostname from your browser address bar.")

## Step 4 — Start the app

The cell below starts chainlit on port 8081 and a small path-rewriting proxy on port 8080 (the notebook proxy port).
After running it, open the proxy URL from Step 3 in your browser.

In [ ]:
import base64
import subprocess
import time

# kill any existing processes from a previous run
subprocess.run(["pkill", "-f", "chainlit run app.py"], capture_output=True)
subprocess.run(["pkill", "-f", "proxy:app"], capture_output=True)
time.sleep(1)

# get the MaaS token
result = subprocess.run(
    ["oc", "get", "secret", "maas-secret", "-o", "jsonpath={.data.token}"],
    capture_output=True, text=True
)
MAAS_TOKEN = base64.b64decode(result.stdout.strip()).decode()

env = os.environ.copy()
env["MAAS_TOKEN"] = MAAS_TOKEN
env["ROOT_PATH"]  = ROOT_PATH

# start chainlit on port 8081 first and wait for it to be ready
chainlit_proc = subprocess.Popen(
    ["chainlit", "run", "app.py", "--port", "8081", "--host", "0.0.0.0", "--root-path", ROOT_PATH],
    env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print("Waiting for chainlit to start...")
time.sleep(6)

if chainlit_proc.poll() is not None:
    print("ERROR: chainlit failed to start")
    print(chainlit_proc.stdout.read().decode())
else:
    # start the path-rewriting proxy on port 8080 only once chainlit is up
    proxy_proc = subprocess.Popen(
        ["python", "proxy.py"],
        env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    time.sleep(2)

    if proxy_proc.poll() is not None:
        print("ERROR: proxy failed to start")
        print(proxy_proc.stdout.read().decode())
    else:
        print(f"chainlit started (pid {chainlit_proc.pid}) on port 8081")
        print(f"proxy started   (pid {proxy_proc.pid}) on port 8080")
        print()
        print("Open the proxy URL from Step 3 in your browser to chat.")
        print()
        print("To stop: chainlit_proc.terminate(); proxy_proc.terminate()")

## Step 5 — Check chainlit logs (if something looks wrong)

Run this cell to read the last output from the chainlit process.

In [ ]:
import os, select

for name, p in [("chainlit", chainlit_proc), ("proxy", proxy_proc)]:
    print(f"--- {name} (alive: {p.poll() is None}) ---")
    if select.select([p.stdout], [], [], 0.3)[0]:
        output = os.read(p.stdout.fileno(), 4096).decode(errors="replace")
        print(output)
    else:
        print("(no new output)")
    print()

## Step 6 — Stop the app

Run this cell when you are done to shut down both processes and free up the ports.

In [ ]:
chainlit_proc.terminate()
proxy_proc.terminate()
print("Chainlit and proxy stopped.")